# 🔬 SAM 2 Segmentation Demo

本 Notebook 演示如何使用 Meta SAM 2.1 进行图像分割。

**覆盖内容：**
- 环境配置与设备检测
- SAM 2 模型加载
- Point Prompt 分割（点击式）
- Box Prompt 分割（框选式）
- 结果可视化与 Mask 后处理

**硬件要求：** Apple Silicon (MPS) 或 NVIDIA GPU (CUDA)

## 1. 环境配置与设备检测

In [ ]:
import cv2
import numpy as np
import torch
import matplotlib.pyplot as plt

plt.rcParams["figure.dpi"] = 100

print(f"PyTorch: {torch.__version__}")
print(f"OpenCV:  {cv2.__version__}")

In [ ]:
from sam2_lab.device import get_device, get_torch_dtype

device = get_device()
dtype = get_torch_dtype(device)

print(f"Device: {device}")
print(f"Dtype:  {dtype}")

## 2. 加载 SAM 2 模型

In [ ]:
from sam2.build_sam import build_sam2
from sam2.sam2_image_predictor import SAM2ImagePredictor

CHECKPOINT = "models/sam2/checkpoints/sam2.1_hiera_tiny.pt"
MODEL_CFG = "configs/sam2.1/sam2.1_hiera_t.yaml"

print("Loading SAM 2.1 Hiera-Tiny ...")
model = build_sam2(MODEL_CFG, CHECKPOINT, device=device)
predictor = SAM2ImagePredictor(model)
print(f"Loaded on {device}")

## 3. Point Prompt 分割

点击图中目标，SAM 2 自动分割该物体。

In [ ]:
IMAGE_PATH = "data/images/sample_01.jpg"

image_bgr = cv2.imread(IMAGE_PATH)
image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(8, 6))
plt.imshow(image_rgb)
plt.title("Original Image")
plt.axis("off")
plt.show()

In [ ]:
predictor.set_image(image_rgb)

CLICK_X, CLICK_Y = 400, 300  # 根据图片调整

input_point = np.array([[CLICK_X, CLICK_Y]])
input_label = np.array([1])

masks, scores, _ = predictor.predict(
    point_coords=input_point,
    point_labels=input_label,
    multimask_output=True,
)

best_idx = np.argmax(scores)
best_mask = masks[best_idx]
best_score = scores[best_idx]

print(f"Best score: {best_score:.3f}")
print(f"Mask area: {best_mask.sum()} px")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

axes[0].imshow(image_rgb)
axes[0].scatter(CLICK_X, CLICK_Y, c='red', s=80, marker='*',
                edgecolors='white', linewidths=1)
axes[0].set_title(f"Input Point ({CLICK_X}, {CLICK_Y})")
axes[0].axis("off")

axes[1].imshow(best_mask, cmap='gray')
axes[1].set_title(f"Mask (score={best_score:.3f})")
axes[1].axis("off")

axes[2].imshow(image_rgb)
axes[2].imshow(best_mask, cmap='jet', alpha=0.4)
axes[2].scatter(CLICK_X, CLICK_Y, c='red', s=80, marker='*',
                edgecolors='white', linewidths=1)
axes[2].set_title("Overlay")
axes[2].axis("off")

plt.tight_layout()
plt.show()

## 4. Box Prompt 分割

在图上画矩形框，SAM 2 分割框内主要物体。

In [ ]:
BOX = {"x1": 100, "y1": 80, "x2": 700, "y2": 600}

predictor.set_image(image_rgb)

input_box = np.array([[BOX["x1"], BOX["y1"], BOX["x2"], BOX["y2"]]])
masks, scores, _ = predictor.predict(
    point_coords=None, point_labels=None,
    box=input_box, multimask_output=True,
)

best_idx = np.argmax(scores)
best_mask_box = masks[best_idx]
best_score_box = scores[best_idx]

print(f"Best box score: {best_score_box:.3f}")
print(f"Mask area: {best_mask_box.sum()} px")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].imshow(best_mask_box, cmap='gray')
axes[0].set_title(f"Box Mask (score={best_score_box:.3f})")
axes[0].axis("off")

axes[1].imshow(image_rgb)
axes[1].imshow(best_mask_box, cmap='jet', alpha=0.4)
rect = plt.Rectangle((BOX["x1"], BOX["y1"]),
                      BOX["x2"]-BOX["x1"], BOX["y2"]-BOX["y1"],
                      fill=False, edgecolor='red', linewidth=2)
axes[1].add_patch(rect)
axes[1].set_title("Overlay + Box")
axes[1].axis("off")

plt.tight_layout()
plt.show()

## 5. Mask 后处理与质量评估

In [ ]:
from sam2_lab.mask.postprocess import postprocess_mask
from sam2_lab.mask.quality import mask_quality_report

processed = postprocess_mask(best_mask, min_area=500, kernel_size=5)

print("Raw mask quality:")
for k, v in mask_quality_report(best_mask).items():
    print(f"  {k}: {v}")

print("\nProcessed mask quality:")
for k, v in mask_quality_report(processed).items():
    print(f"  {k}: {v}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(best_mask, cmap='gray')
axes[0].set_title(f"Raw ({best_mask.sum()} px)")
axes[0].axis("off")
axes[1].imshow(processed, cmap='gray')
axes[1].set_title(f"Processed ({processed.sum()} px)")
axes[1].axis("off")
plt.tight_layout()
plt.show()

## 6. 小结

| 提示方式 | 使用场景 | 优点 |
|----------|----------|------|
| Point | 点击目标 | 快捷，适合单物体 |
| Box | 框选区域 | 更精确，减少歧义 |

下一步：
- `scripts/04_segment_auto.py` — 全自动分割
- `scripts/07_video_track.py` — 视频追踪
- `scripts/06_sam2_inpaint.py` — SAM2 + Diffusers
- `make api` — RESTful API